# A2 — Cloud Misconfiguration Rule Catalog (L2)

Shallow-clones Prowler, Checkov and ScoutSuite, extracts every check definition, normalises them into one schema, and writes `data/processed/l2_misconfig_catalog.csv`. All logic lives in `src/misconfig/`; this notebook is a thin runner (CLAUDE.md rule 9).

**Severity caveat (matters for A3):** Prowler ships a full five-level severity, ScoutSuite only `danger`/`warning`, and Checkov OSS ships none at all (severities live in the commercial Bridgecrew platform). See the mapping documented in `src/misconfig/normaliser.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.misconfig.normaliser import build_l2_catalog, log_stats

In [ ]:
df = build_l2_catalog()
log_stats(df)

In [ ]:
df.head()

## Cross-tool coverage — what A3 has to work with

A3 compares severities across tools, so it needs checks that more than one tool covers. Checkov contributes no severity, which constrains that analysis to Prowler vs ScoutSuite unless a proxy is used.

In [ ]:
coverage = (
    df[df.provider == "aws"]
    .groupby("service")
    .agg(tools=("tool", "nunique"), checks=("catalog_id", "count"))
    .sort_values(["tools", "checks"], ascending=False)
)
coverage.head(15)